Grupo 4 - 19/05/2026 - IA - Trabajo IA - Extracción de Embeddings

# Notebook 3: Extracción de Embeddings Faciales

En el Notebook 2 detectamos y recortamos todas las caras del dataset del Gobierno de La Rioja, normalizándolas a 160×160 px. En este notebook usaremos estos recortes para generar embeddings faciales. En este notebook buscaremos comparar tres modelos de reconocimiento facial(VGGFace, OpenFace y FaceNet) y elegir el mejor de estos y finalmente generaremos los embeddings para todos los recortes del dataset.

## 0. Carga de librerías

Cargamos las librerías necesarias para la deteccion.

In [1]:
%%capture
!pip install deepface tf-keras

In [2]:
import os
import cv2
import pickle
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from deepface import DeepFace
%matplotlib inline

26-05-19 09:05:20 - Directory /root/.deepface has been created
26-05-19 09:05:20 - Directory /root/.deepface/weights has been created


## 1. Carga del Dataset de Caras

Descargamos el ZIP con los recortes generados en el Notebook 2 desde el release de GitHub.

In [3]:
%%capture
!wget FACE_CROPS_URL -O carasRecortadas.zip
!unzip carasRecortadas.zip -d carasRecortadas

In [4]:
datasetPath = Path('/content/carasRecortadas')

caras = sorted(datasetPath.glob('*.jpg'))

print(f"Total de imágenes: {len(caras)}")

Total de imágenes: 38299


## 2. ¿Qué es un Embedding Facial?

Un embedding facial es un vector numérico de dimensión fija que representa la identidad de una cara. El modelo transforma la imagen de entrada en ese vector de forma que:

- Dos imágenes de la misma persona producen vectores próximos entre sí.
- Dos imágenes de personas distintas producen vectores lejanos.

Es esta propiedad es la que nos permite construir un motor de búsqueda facial.

## 3. Comparativa de Modelos

Comparamos tres modelos de reconocimiento facial disponibles en DeepFace:

| Modelo | Dimensión embedding | Arquitectura |
|---|---|---|
| **VGG-Face** | 4096 | VGGNet |
| **OpenFace** | 128 | SqueezeNet |
| **FaceNet** | 128 | Inception |

Los evaluamos sobre una muestra de 1000 recortes midiendo tiempo y dimensión del embedding resultante.

In [5]:
def rendimiento_modelo(modelo, muestra):
    t0 = time.time()
    embeddings = [
        DeepFace.represent(img_path=str(ruta), model_name=modelo, enforce_detection=False)[0]['embedding']
        for ruta in muestra
    ]
    tf = time.time() - t0
    print(f"{modelo:<15} dim={len(embeddings[0]):<6} {tf:.2f}s")
    return embeddings

In [6]:
random.seed(42)
muestra = random.sample(caras, 1000)

### 3.1 Modelo 1 : VGG-Face

In [7]:
embeddings_vgg = rendimiento_modelo('VGG-Face', muestra)

26-05-19 09:05:40 - 🔗 vgg_face_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5 to /root/.deepface/weights/vgg_face_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5
To: /root/.deepface/weights/vgg_face_weights.h5
100%|██████████| 580M/580M [00:06<00:00, 90.8MB/s]


VGG-Face        dim=4096   82.67s


### 3.2 Modelo 2 : OpenFace

In [8]:
embeddings_openface = rendimiento_modelo('OpenFace', muestra)

26-05-19 09:07:02 - 🔗 openface_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/openface_weights.h5 to /root/.deepface/weights/openface_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/openface_weights.h5
To: /root/.deepface/weights/openface_weights.h5
100%|██████████| 15.3M/15.3M [00:00<00:00, 118MB/s] 


OpenFace        dim=128    136.03s


### 3.3 Modelo 3 : Facenet

In [9]:
embeddings_facenet = rendimiento_modelo('Facenet', muestra)

26-05-19 09:09:19 - 🔗 facenet_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5 to /root/.deepface/weights/facenet_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: /root/.deepface/weights/facenet_weights.h5
100%|██████████| 92.2M/92.2M [00:00<00:00, 128MB/s]


Facenet         dim=128    280.24s


### 3.4 Resultados

| Modelo    | Dimensión | Tiempo  | imgs/s |
|-----------|----------:|-------------------:|-------:|
| VGG-Face  | 4096      | 82.67s             | 12.1   |
| OpenFace  | 128       | 136.03s            | 7.4    |
| Facenet   | 128       | 280.24s            | 3.6    |

El resultado mas raro es que VGG-Face sea el más rápido aun produciendo los embeddings más grandes. Buscandolo, esto se explica por que es el que lleva mas año de soporte y el que esta mejor optimizado para la GPU de google colab

Para el producto final, la elección del modelo depende de dos cosas:
- la calidad de los embeddings como representación de identidad
- el coste de búsqueda en el VP-Tree del Notebook 4.

En cuanto a la búsqueda, para el VP-Tree resulta mucho más eficiente trabajar con 128 dimensiones que con 4096, especialmente al manejar 38.299 elementos. Por lo tanto, OpenFace y FaceNet parten con ventaja, haciendo que el motor de búsqueda sea mucho más rápido y ligero.Por otro lado aunque OpenFace y FaceNet producen vectores de 128 dimensiones, FaceNet nos da mayor precisión porque se ha entrenado con bases de datos más grandes. Si bien toma más tiempo que OpenFace, esto es solo un proceso que se hace una vez y no ralentiza las búsquedas posteriores. Por tanto, FaceNet es la mejor opción.

## 4. Generación de embeddings sobre el dataset completo

Vamos a generar un embedding FaceNet de 128 dimensiones para cada recorte del dataset. El resultado se almacenara en un diccionario `{nombre_fichero: vector}` que guardaremos en `embeddings.pkl`.

Al igual que en los notebooks anteriores, ejecutaremos este proceso localmente usando la GPU para poder procesar todo el dataset en un tiempo razonable.

In [4]:
!pip install tf-keras

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------------- -- 1.6/1.7 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 6.9 MB/s  0:00:00


In [1]:
%%capture
!pip install deepface
!pip install deepface tensorflow[and-cuda]

In [2]:
import time
import pickle
import numpy as np
from pathlib import Path
from deepface import DeepFace
from collections import defaultdict


26-05-19 11:48:52 - Directory <local-path> has been created
26-05-19 11:48:52 - Directory <local-path> has been created


In [3]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

In [4]:
carasPath = Path('carasRecortadas')
caras = [p for p in carasPath.rglob('*.jpg')]
print(f"Caras encontradas: {len(caras)}")

Caras encontradas: 38299


In [5]:
DeepFace.build_model('Facenet')
print("Modelo Facenet cargado en GPU.")

26-05-19 11:48:55 - 🔗 facenet_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5 to <local-path>


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: <local-path>
100%|█████████████████████████████████████████████████████████████████████████████| 92.2M/92.2M [00:03<00:00, 23.2MB/s]


Modelo Facenet cargado en GPU.


In [6]:
embeddings = {}
contadores = defaultdict(int)

print(f"Procesando {len(caras)} caras...")
t_inicio = time.time()

for idx, ruta in enumerate(caras):
    try:
        emb = DeepFace.represent(
            img_path=str(ruta),
            model_name='Facenet',
            enforce_detection=False
        )[0]['embedding']
        embeddings[ruta.name] = emb
        contadores['ok'] += 1
    except Exception:
        contadores['error'] += 1

    if idx % 1000 == 0:
        print(f"  {idx}/{len(caras)} — {contadores['ok']} embeddings...")

tf = time.time() - t_inicio
print(f"\nCompletado en {tf:.1f}s ({tf/60:.1f} min)")
print(f"  Embeddings generados : {contadores['ok']}")
print(f"  Errores              : {contadores['error']}")

Procesando 38299 caras...
  0/38299 — 1 embeddings...
  1000/38299 — 1001 embeddings...
  2000/38299 — 2001 embeddings...
  3000/38299 — 3001 embeddings...
  4000/38299 — 4001 embeddings...
  5000/38299 — 5001 embeddings...
  6000/38299 — 6001 embeddings...
  7000/38299 — 7001 embeddings...
  8000/38299 — 8001 embeddings...
  9000/38299 — 9001 embeddings...
  10000/38299 — 10001 embeddings...
  11000/38299 — 11001 embeddings...
  12000/38299 — 12001 embeddings...
  13000/38299 — 13001 embeddings...
  14000/38299 — 14001 embeddings...
  15000/38299 — 15001 embeddings...
  16000/38299 — 16001 embeddings...
  17000/38299 — 17001 embeddings...
  18000/38299 — 18001 embeddings...
  19000/38299 — 19001 embeddings...
  20000/38299 — 20001 embeddings...
  21000/38299 — 21001 embeddings...
  22000/38299 — 22001 embeddings...
  23000/38299 — 23001 embeddings...
  24000/38299 — 24001 embeddings...
  25000/38299 — 25001 embeddings...
  26000/38299 — 26001 embeddings...
  27000/38299 — 27001 embedd

In [7]:
OUT = Path('embeddings.pkl')
with open(OUT, 'wb') as f:
    pickle.dump(embeddings, f)
print(f"{OUT} guardado correctamente ")

embeddings.pkl guardado correctamente 
